# 逐事件軌跡動畫(replay_epoch)

`example/replay_epoch.py` 用某個 epoch 存的權重、強制 `chunk_size=1` 重跑
`run_network_traced`,拿到逐事件精確的 `LayerForwardTrace`(`spike_mask`/
`v_steps`/`event_ms`,3 欄,見該檔案 docstring)。這份 notebook 把它依
`event_ms` 攤成真實時間軸播放成動畫:離散量(`spike_mask`)用瞬時顯示
(`viz.time_resample.resample_pulse`——spike 是瞬時事件,事件跟事件之間這個
neuron 根本沒被計算過,不是「還在 fire」,所以沒有事件的 frame 一律顯示沒
fire,不會沿用前一筆事件的值),連續量(`v_steps`)用衰減插值(`resample_decay`,
用該層的 `tau`,電位是真的連續存在的物理量,沿用前一筆事件的值再衰減才有
意義)——插值/動畫繪圖邏輯都在 `viz/`,這裡只負責讀資料、組裝、擺 UI。

conv 層跟 FC 層呈現方式不一樣(conv 攤成 `(oc,h,w)` 圖排網格;FC 沒有空間
結構,改成選 neuron 範圍 x 滑動時間窗口),各自「怎麼呈現自己」收進
`viz/replay_panels.py` 的 `ConvChannelPanel`/`FCWindowPanel`——這是可重用的
轉接層(只依賴 `salt_core`+`viz`,跟這個 notebook/`example/` 無關),這裡只
負責組裝、擺 UI,不重複寫換算邏輯。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

from example.paths import EXPERIMENTS_DIR
from example.replay_epoch import _load_train_sample, load_epoch_weights, replay_sample
from example.utils import load_run_record
from salt_core.layers import ConvLayer, FCLayer
from viz.channel_grid import ChannelGridAnimation
from viz.replay_panels import ConvChannelPanel, FCWindowPanel
from viz.time_resample import build_frame_grid


## 參數

改這裡,不改函式庫。

In [ ]:
RUN_NAME = "conv_compressed_compressed_maxsteps_verify_20260913_120425"
EXP_DIR = str(EXPERIMENTS_DIR / RUN_NAME)
EPOCH = 0            # 要有對應的 weights/epoch_XXX.npz
DT_MS = 1.0          # 動畫的時間格解析度(毫秒),越小動畫越平滑但 frame 數越多
N_CONV_PANELS = 2    # 要幾個 conv 窗口(全部擠在同一排,見下面組裝)
N_FC_PANELS = 1      # 要幾個 FC 窗口(各自一排)

# 每一排的高度比重,對應 viz.channel_grid.ChannelGridAnimation.add_row 的
# height 參數——兩個都是 1.0 就是等分;FC 窗口的圖天生比較扁,預設給矮一點。
# 兩個都可以調,沒有誰只能被誰決定。
CONV_ROW_HEIGHT = 1.0
FC_ROW_HEIGHT = 0.6


## 讀這個 epoch 的層(幾何/`tau` 只跟 epoch 有關,跟 sample 無關,只需要讀一次)

conv 層跟 FC 層分開放進兩份選單,因為呈現方式完全不同(conv 攤成 `(oc,h,w)`
圖排網格;FC 沒有空間結構,見下面「FC 層」那節)。

In [9]:
LAYERS, _ = load_epoch_weights(EXP_DIR, EPOCH)
RUN_RECORD = load_run_record(EXP_DIR)

CONV_LAYERS = [l for l in LAYERS if isinstance(l, ConvLayer)]
CONV_LAYER_NAMES = [l.name for l in CONV_LAYERS]
print("conv 層:", CONV_LAYER_NAMES)

FC_LAYERS = [l for l in LAYERS if isinstance(l, FCLayer)]
FC_LAYER_NAMES = [l.name for l in FC_LAYERS]
print("FC 層:", FC_LAYER_NAMES, [l.n_neurons for l in FC_LAYERS])


conv 層: ['conv1', 'conv2']
FC 層: ['out'] [10]


## 組裝:conv 跟 FC 一起比較

用法分三步,對應下面三個 cell:

1. **資料快取**:執行一次就好,提供 `_trace_for(sample, layer_name)` 給下面
   兩步用,自己不用改。
2. **表單**:每一列(conv/FC 各 `N_CONV_PANELS`/`N_FC_PANELS` 列,上面
   「參數」設)獨立填 `sample`/`layer`/`quantity`/... ,填完不用重新執行
   這格。
3. **畫**:按「畫」才會真的照表單目前的值重跑 forward、組裝動畫——每次按都
   是重新讀一次表單,可以改表單、再按一次,不用重新執行前兩步。

排版用 `viz.channel_grid.ChannelGridAnimation` 的
`add_row(*panels, height=..., widths=...)`:呼叫一次加一排,`height` 決定
這排多高、`widths` 決定排裡每個 panel 多寬(都不給就是等分,見上面「參數」
的 `CONV_ROW_HEIGHT`/`FC_ROW_HEIGHT`),排跟排彼此獨立、互不影響。畫面排法
固定是 `Conv Conv / FC / FC`:conv 窗口全部擠在同一排、FC 窗口各自一排——
這是直接寫兩次 `add_row`(conv 一次、FC 每個各一次),不是什麼通用排版演算
法,想要的排法跟現在不一樣就直接改這兩行怎麼呼叫 `add_row`,不用改
`viz/`。`ConvChannelPanel`/`FCWindowPanel`(見 `viz/replay_panels.py`)只
負責描述「要畫什麼」,`viz.channel_grid` 完全不知道 conv/FC 是什麼,只依照
這裡呼叫 `add_row` 的順序跟參數排版。

In [10]:
# 步驟 1/3:資料快取。換 sample/epoch 才需要重跑 forward,所以用
# (epoch, sample) 當 key 快取起來——只留最新一筆,不無限累積記憶體。
_traces_cache = {}


def _get_traces(sample: int):
    key = (EPOCH, sample)
    if key not in _traces_cache:
        _traces_cache.clear()
        raw_sample = _load_train_sample(RUN_RECORD, sample)
        _traces_cache[key] = replay_sample(EXP_DIR, EPOCH, *raw_sample)
    return _traces_cache[key]


def _trace_for(sample: int, layer_name: str):
    idx = next(i for i, l in enumerate(LAYERS) if l.name == layer_name)
    return _get_traces(sample)[idx]


In [11]:
# 步驟 2/3:表單。這裡的「一列」是 ipywidgets 的輸入列(一組
# sample/layer/quantity/... 的欄位),跟下面 viz.channel_grid 排版用的
# 「一排 panel」是兩個完全不同的概念,所以特意叫 _spec_form 不叫 _row,
# 避免搞混。填完不用重新執行這格,按「畫」(下一格)才會讀。
def _conv_spec_form(default_channel: int = 0):
    return widgets.HBox([
        widgets.IntText(value=0, description="sample", layout=widgets.Layout(width="150px")),
        widgets.Dropdown(options=CONV_LAYER_NAMES, description="layer", layout=widgets.Layout(width="190px")),
        widgets.Dropdown(options=["spike_mask", "v_steps"], description="quantity",
                          layout=widgets.Layout(width="220px")),
        widgets.IntText(value=default_channel, description="channel", layout=widgets.Layout(width="150px")),
    ])


def _fc_spec_form(default_hi: int = 10):
    return widgets.HBox([
        widgets.IntText(value=0, description="sample", layout=widgets.Layout(width="150px")),
        widgets.Dropdown(options=FC_LAYER_NAMES, description="layer", layout=widgets.Layout(width="190px")),
        widgets.Dropdown(options=["spike_mask", "v_steps"], description="quantity",
                          layout=widgets.Layout(width="220px")),
        widgets.IntText(value=0, description="neuron lo", layout=widgets.Layout(width="150px")),
        widgets.IntText(value=default_hi, description="neuron hi", layout=widgets.Layout(width="150px")),
        widgets.FloatText(value=10.0, description="window±ms", layout=widgets.Layout(width="160px")),
    ])


_conv_forms = [_conv_spec_form() for _ in range(N_CONV_PANELS)]
_fc_default_hi = min(10, FC_LAYERS[0].n_neurons) if FC_LAYERS else 10
_fc_forms = [_fc_spec_form(_fc_default_hi) for _ in range(N_FC_PANELS)]

In [ ]:
# 步驟 3/3:畫。按一次「畫」才會照表單目前的值重跑 forward、組裝動畫——
# 可以改表單、再按一次,不用重新執行前兩格。
def _build_grid_and_frame_ms():
    conv_specs = [tuple(child.value for child in form.children) for form in _conv_forms]
    fc_specs = [tuple(child.value for child in form.children) for form in _fc_forms]

    t_end = 0.0
    for sample, layer_name, _, _ in conv_specs:
        t_end = max(t_end, float(np.nanmax(np.asarray(_trace_for(sample, layer_name).event_ms))))
    for sample, layer_name, _, _, _, _ in fc_specs:
        t_end = max(t_end, float(np.nanmax(np.asarray(_trace_for(sample, layer_name).event_ms))))
    frame_ms = build_frame_grid(0.0, t_end, DT_MS)

    conv_panels = [
        ConvChannelPanel(next(l for l in CONV_LAYERS if l.name == layer_name), channel, quantity,
                          _trace_for(sample, layer_name), frame_ms, DT_MS, label_prefix=f"s{sample} / ")
        for sample, layer_name, quantity, channel in conv_specs
    ]

    # 畫面排法固定是 Conv Conv / FC / FC:conv 全部擠在同一排,FC 每個各自
    # 一排。就這兩種排法,直接寫,不用迴圈湊。
    grid = ChannelGridAnimation()
    grid.add_row(*conv_panels, height=CONV_ROW_HEIGHT)
    for sample, layer_name, quantity, neuron_lo, neuron_hi, window_ms in fc_specs:
        layer = next(l for l in FC_LAYERS if l.name == layer_name)
        trace = _trace_for(sample, layer_name)
        panel = FCWindowPanel(layer, neuron_lo, neuron_hi, quantity, window_ms,
                               trace, frame_ms, DT_MS, label_prefix=f"s{sample} / ")
        grid.add_row(panel, height=FC_ROW_HEIGHT)
    return grid, frame_ms


_go_button = widgets.Button(description="畫")
_go_output = widgets.Output()


def _on_go_clicked(_):
    with _go_output:
        _go_output.clear_output(wait=True)
        grid, frame_ms = _build_grid_and_frame_ms()
        anim = grid.build(frame_labels=[f"t = {ms:.1f} ms" for ms in frame_ms])
        plt.close(anim._fig)  # 避免 %matplotlib inline 把靜態圖跟 jshtml 動畫重複顯示
        display(HTML(anim.to_jshtml()))


_go_button.on_click(_on_go_clicked)

display(widgets.VBox(
    [widgets.HTML("<b>conv 窗口</b>")] + _conv_forms +
    [widgets.HTML("<b>FC 窗口</b>")] + _fc_forms
))

display(_go_button, _go_output)


Button(description='畫', style=ButtonStyle())

Output()